# Phase 4-C｜First Real Research End-to-End Acceptance（只讀評估）

**Notebook 版本：`P4C-EVAL-v1`**　｜　Evaluator pin `883d14c`

`既有真實 Archive → Archive Normalizer → Research Evaluator → Research Decision → Immutable Evaluation Archive`

- 來源：Phase 4-A run `20261107_081300_9439f8c`（授權 1/1 已耗盡）。**本 notebook 不重新回測、不安裝 Core、不讀 Snapshot 或 Holdout、不消耗任何額度、不寫 governance/registry。**
- Archive `MANIFEST.json` 的 SHA 取自 PR #5 中已獨立提交的副本（`docs/evidence/phase4a_dev_smoke/…/MANIFEST.json`，commit `d456fed`），**不是** dataset manifest SHA。
- 所有計算都在 `src/strategy_agent/evaluation/phase4c.py`；notebook 只是 thin runner。
- 輸出：`MyDrive/Quant_Research/strategy-research-agent/evaluations/<YYYYMMDD_HHMMSS>_<evaluator7>/`（8 個檔案，不可覆寫；驗收失敗也會留下診斷）。

請在**新的 Colab runtime** 執行 Run all（C8 會確認這個 process 沒有載入任何策略執行器）。

In [ ]:
# --- 1. 設定（不要修改 pin）---
NOTEBOOK_VERSION = "P4C-EVAL-v1"
NOTEBOOK_PATH    = "notebooks/Phase4C_Real_Evaluation_Acceptance.ipynb"
AGENT_REPO    = "https://github.com/hh4832/strategy-research-agent.git"
BRANCH        = "feature/phase4c-real-e2e-acceptance"
EVALUATOR_SHA = "883d14c0059add270d596e4bdc86be46052219ea"
SOURCE_RUN_ID = "20261107_081300_9439f8c"
SOURCE_ARCHIVE = "/content/drive/MyDrive/Quant_Research/strategy-research-agent/" + SOURCE_RUN_ID
ARCHIVE_MANIFEST_SHA = "328f2cf99c33b4d9d427327d092aaf31da619b5f5ad948c815e75b04b92345a9"   # MANIFEST.json of the archive
DATASET_MANIFEST_SHA = "8b6fc838b72ae3015f6eb418c3a77115e53a86f185e05c2249e9dcd58192cfc5"   # a DIFFERENT file (snapshot)
EVAL_ROOT  = "/content/drive/MyDrive/Quant_Research/strategy-research-agent/evaluations"
AGENT_ROOT = "/content/strategy-research-agent-phase4c"

import json, os, re, subprocess, sys, hashlib
from pathlib import Path
assert sys.version_info >= (3, 11), sys.version
assert ARCHIVE_MANIFEST_SHA != DATASET_MANIFEST_SHA
SOURCE_ARCHIVE, EVAL_ROOT, AGENT_ROOT = Path(SOURCE_ARCHIVE), Path(EVAL_ROOT), Path(AGENT_ROOT)
def git(*a, cwd=None):
    r = subprocess.run(["git", *a], cwd=cwd, text=True, capture_output=True)
    if r.returncode: raise RuntimeError("git " + a[0] + " failed: " + r.stderr[-400:])
    return r.stdout.strip()
print(NOTEBOOK_VERSION, "| evaluator pin", EVALUATOR_SHA[:7])

In [ ]:
# --- 2. Version Sync ---
stale = [m for m in ("strategy_agent", "market_research_core") if m in sys.modules]
assert not stale, f"STOP: runtime 已載入 {stale}（可能是舊 notebook 或其他 session）；請 Runtime → Restart session 後 Run all"
if not AGENT_ROOT.exists():
    git("clone", "--quiet", AGENT_REPO, str(AGENT_ROOT)); git("checkout", "--quiet", "--detach", EVALUATOR_SHA, cwd=AGENT_ROOT)
    print("Fresh checkout created")
else:
    print("Existing checkout detected: verifying before reuse")
remote = git("remote", "get-url", "origin", cwd=AGENT_ROOT)
assert remote.rstrip("/").removesuffix(".git") == AGENT_REPO.removesuffix(".git"), "STOP: wrong remote " + remote
head = git("rev-parse", "HEAD", cwd=AGENT_ROOT)
status = git("status", "--porcelain", "--untracked-files=all", cwd=AGENT_ROOT)
git("fetch", "--quiet", "origin", BRANCH, cwd=AGENT_ROOT)
remote_head = git("rev-parse", "FETCH_HEAD", cwd=AGENT_ROOT)
on_branch = subprocess.run(["git", "merge-base", "--is-ancestor", EVALUATOR_SHA, remote_head], cwd=AGENT_ROOT).returncode == 0
nb = subprocess.run(["git", "show", f"{remote_head}:{NOTEBOOK_PATH}"], cwd=AGENT_ROOT, text=True, capture_output=True)
gh_version = (re.search(r'NOTEBOOK_VERSION = \\"([^\\"]+)\\"', nb.stdout) or [None, None])[1] if nb.returncode == 0 else None
SYNC = {"notebook_version": NOTEBOOK_VERSION, "notebook_version_on_github_head": gh_version, "github_remote_head": remote_head,
        "checkout_head": head, "pinned_evaluator_sha": EVALUATOR_SHA, "pin_on_branch": on_branch,
        "working_tree_clean": status == "", "working_tree_status": status.splitlines()[:10]}
print(json.dumps(SYNC, indent=2))
assert head == EVALUATOR_SHA, "STOP: checkout HEAD ≠ pin（不自動 checkout/reset；請換 AGENT_ROOT 或重啟 runtime）"
assert status == "", "STOP: checkout 有變更"
assert on_branch, "STOP: pin 不在 GitHub 分支上"
assert gh_version in (None, NOTEBOOK_VERSION), f"STOP: 這是舊 notebook（{NOTEBOOK_VERSION}），GitHub 上是 {gh_version}；請從 GitHub 重新開啟"
if remote_head != EVALUATOR_SHA:
    print("NOTE: GitHub HEAD 比 pin 新；仍以 pin 執行。變更：",
          git("diff", "--name-only", EVALUATOR_SHA, remote_head, cwd=AGENT_ROOT).splitlines() or "none")
# Evaluator only: no Core, no executor dependencies.
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(AGENT_ROOT)], check=True)
sys.path.insert(0, str(AGENT_ROOT / "src"))
from strategy_agent.evaluation import phase4c as C
IMPORTS = {"phase4c": C.__file__}
assert C.__file__.startswith(str(AGENT_ROOT / "src")), "STOP: evaluator 不是從 pinned checkout 載入"
assert "market_research_core" not in sys.modules
# Independent pin: the MANIFEST.json copy committed in d456fed must hash to the pin.
committed = AGENT_ROOT / "docs/evidence/phase4a_dev_smoke" / SOURCE_RUN_ID / "MANIFEST.json"
assert hashlib.sha256(committed.read_bytes()).hexdigest() == ARCHIVE_MANIFEST_SHA, "STOP: committed MANIFEST copy ≠ pin"
SYNC |= {"import_source": IMPORTS, "core_installed_or_imported": False}
print("VERSION SYNC: PASS | import", IMPORTS)

In [ ]:
# --- 3. Drive（只讀載入既有 Archive）---
from google.colab import drive
drive.mount("/content/drive")
assert SOURCE_ARCHIVE.is_dir(), "STOP: source archive not found " + str(SOURCE_ARCHIVE)
listing = sorted(p.name for p in SOURCE_ARCHIVE.iterdir())
print(len(listing), "files:", listing)
SOURCE_BEFORE = C.fingerprint(SOURCE_ARCHIVE)
assert len(listing) == 14, "STOP: archive must contain exactly the 14 contract files"

In [ ]:
# --- 4. Archive integrity（MANIFEST pin → 13 個檔案 SHA）---
arch = C.read_archive(SOURCE_ARCHIVE, expected_manifest_sha256=ARCHIVE_MANIFEST_SHA)
print("MANIFEST.json pin OK; files verified:", len(arch["files"]), "| run", arch["manifest"]["run_id"])
del arch

In [ ]:
# --- 5. Evaluator + acceptance → immutable evaluation archive ---
RESULT = C.run(SOURCE_ARCHIVE, expected_manifest_sha256=ARCHIVE_MANIFEST_SHA, output_root=EVAL_ROOT,
               expected_evaluator_sha=EVALUATOR_SHA, governance_root=AGENT_ROOT,
               context={"notebook": NOTEBOOK_VERSION, "version_sync": SYNC})
print(json.dumps({k: RESULT[k] for k in ("status", "checks", "engineering_status", "strategy_verdict",
                                        "evidence_sufficiency", "output_dir", "evaluator_commit", "evaluated_at")}, indent=2))
assert C.fingerprint(SOURCE_ARCHIVE) == SOURCE_BEFORE, "SOURCE ARCHIVE CHANGED"
print("source archive unchanged:", True)

In [ ]:
# --- 6. Acceptance report + 下載 ---
import zipfile
OUT = Path(RESULT["output_dir"])
acc = json.loads((OUT / "acceptance_checks.json").read_text())
for k, v in acc["checks"].items():
    print(f"{k:32s} {v['status']}")
if (OUT / "research_decision.json").exists():
    print(json.dumps(json.loads((OUT / "research_decision.json").read_text()), indent=2, ensure_ascii=False)[:2000])
zp = Path("/content") / (OUT.name + "_phase4c_evaluation.zip")
with zipfile.ZipFile(zp, "w") as z:
    for p in sorted(OUT.iterdir()): z.write(p, p.name)
from google.colab import files
files.download(str(zp))
print("Evaluation archive:", OUT)